# 🏏 Cricket Shot Classification using Machine Learning
**Dataset:** [Cricket Shot Dataset — Kaggle (aneesh10)](https://www.kaggle.com/datasets/aneesh10/cricket-shot-dataset/data)  
**Target Classes:** Pull Shot | Leg Glance-Flick | Drive | Sweep  
**GitHub:** https://github.com/users/Najam000/projects/1  
**Streamlit Cloud:** https://cricket-shot-classifier.streamlit.app

---

## 1. Imports & Setup

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import os
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Image Processing ─────────────────────────────────────────────────────────
from PIL import Image
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize

# ── Sklearn Preprocessing ────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, StratifiedKFold)

# ── Sklearn Models ───────────────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# ── Metrics ──────────────────────────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

# ── Display Settings ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print('✅ All imports successful!')
print(f'Working directory: {os.getcwd()}')

## 2. Load Dataset

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Dataset Structure (after downloading from Kaggle):
#   cricket-shot-dataset/
#     ├── pull_shot/       (~1000 images)
#     ├── leg_glance_flick/ (~1000 images)
#     ├── drive/           (~1000 images)
#     └── sweep/           (~1000 images)
# ──────────────────────────────────────────────────────────────────────────────

DATASET_DIR = './cricket-shot-dataset'   # ← update if your path differs
IMG_SIZE    = (64, 64)                   # resize all images to 64×64
CLASSES     = ['pull_shot', 'leg_glance_flick', 'drive', 'sweep']

def extract_hog_features(img_array, img_size=(64, 64)):
    """Convert image → grayscale → resize → HOG feature vector."""
    img_resized = resize(img_array, img_size, anti_aliasing=True)
    if img_resized.ndim == 3:
        img_gray = rgb2gray(img_resized)
    else:
        img_gray = img_resized
    features = hog(img_gray,
                   orientations=9,
                   pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2),
                   feature_vector=True)
    return features

def load_dataset(dataset_dir, classes, img_size=(64, 64)):
    """Walk dataset folder, extract HOG features, return X and y arrays."""
    X, y = [], []
    class_counts = {}

    for label, cls in enumerate(classes):
        cls_dir = os.path.join(dataset_dir, cls)
        if not os.path.isdir(cls_dir):
            print(f'⚠️  Folder not found: {cls_dir}')
            continue

        count = 0
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                fpath = os.path.join(cls_dir, fname)
                try:
                    img = np.array(Image.open(fpath).convert('RGB'))
                    feat = extract_hog_features(img, img_size)
                    X.append(feat)
                    y.append(label)
                    count += 1
                except Exception as e:
                    print(f'  Skip {fname}: {e}')
        class_counts[cls] = count
        print(f'  {cls:<22} → {count} images loaded')

    return np.array(X), np.array(y), class_counts

print('📂 Loading Cricket Shot Dataset …')
X, y, class_counts = load_dataset(DATASET_DIR, CLASSES, IMG_SIZE)
print(f'\n✅ Dataset loaded: {X.shape[0]} samples, {X.shape[1]} HOG features each')
print(f'   Classes: {CLASSES}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3a. Class Distribution ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = list(class_counts.values())
labels = [c.replace('_', ' ').title() for c in class_counts.keys()]

axes[0].bar(labels, counts, color=sns.color_palette('Set2', 4), edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].set_xlabel('Shot Type')
for i, v in enumerate(counts):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

axes[1].pie(counts, labels=labels, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', 4), startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nDataset Summary:')
print(f'  Total images  : {sum(counts)}')
print(f'  Unique classes: {len(CLASSES)}')
print(f'  HOG feat size : {X.shape[1]}')

In [ ]:
# ── 3b. Sample Images per Class ───────────────────────────────────────────────
fig, axes = plt.subplots(4, 5, figsize=(16, 12))
fig.suptitle('Sample Cricket Shot Images per Class', fontsize=16, fontweight='bold')

for row, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATASET_DIR, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    sample_imgs = np.random.choice(imgs, min(5, len(imgs)), replace=False)
    for col, fname in enumerate(sample_imgs):
        img = mpimg.imread(os.path.join(cls_dir, fname))
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(cls.replace('_',' ').title(),
                                      fontsize=10, fontweight='bold',
                                      rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.savefig('eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3c. HOG Feature Statistics ────────────────────────────────────────────────
df_feat = pd.DataFrame({'class': y,
                         'mean_hog': X.mean(axis=1),
                         'std_hog':  X.std(axis=1),
                         'max_hog':  X.max(axis=1)})
class_name_map = {i: c.replace('_',' ').title() for i, c in enumerate(CLASSES)}
df_feat['class_name'] = df_feat['class'].map(class_name_map)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in zip(axes,
                           ['mean_hog', 'std_hog', 'max_hog'],
                           ['Mean HOG Value', 'Std HOG Value', 'Max HOG Value']):
    for cls_name in df_feat['class_name'].unique():
        vals = df_feat[df_feat['class_name']==cls_name][col]
        ax.hist(vals, bins=30, alpha=0.6, label=cls_name)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_hog_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_feat.groupby('class_name')[['mean_hog','std_hog','max_hog']].describe().round(4))

## 4. Preprocessing

In [ ]:
# ── Train/Test Split (80/20) ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')

# ── StandardScaler — fit on TRAIN only (avoid data leakage) ──────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform train
X_test_sc  = scaler.transform(X_test)        # transform test only

print(f'Feature mean after scaling: {X_train_sc.mean():.4f}  (≈ 0.0)')
print(f'Feature std  after scaling: {X_train_sc.std():.4f}   (≈ 1.0)')
print('✅ Preprocessing complete — no data leakage!')

## 5. Train & Compare All Models (Baseline)

In [ ]:
# Quick comparison of multiple algorithms before tuning
models = {
    'SVM (RBF)'           : SVC(kernel='rbf', random_state=42, probability=True),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Extra Trees'         : ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors' : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'         : GaussianNB(),
}

results = []
print(f'{"Model":<25} {"Train Acc":>10} {"Test Acc":>10}')
print('─' * 50)

for name, clf in models.items():
    clf.fit(X_train_sc, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train_sc))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test_sc))
    results.append({'Model': name, 'Train Accuracy': train_acc, 'Test Accuracy': test_acc})
    print(f'{name:<25} {train_acc:>10.4f} {test_acc:>10.4f}')

df_results = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print('\nTop 3 Models:')
print(df_results.head(3).to_string(index=False))

In [ ]:
# ── Visualise Baseline Comparison ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_results))
w = 0.35
bars1 = ax.bar(x - w/2, df_results['Train Accuracy'], w, label='Train Acc',
                color='steelblue', alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + w/2, df_results['Test Accuracy'],  w, label='Test Acc',
                color='coral',    alpha=0.85, edgecolor='black')

for b in bars2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(df_results['Model'], rotation=25, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Baseline Model Comparison — Cricket Shot Classification',
             fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cross-Validation (Top 3 Models)

5-fold CV gives a more reliable performance estimate

In [ ]:
top3_names = df_results['Model'].head(3).tolist()
top3_models = {n: models[n] for n in top3_names}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
print('5-Fold Cross-Validation Results:')
print('─' * 55)

for name, clf in top3_models.items():
    scores = cross_val_score(clf, X_train_sc, y_train, cv=cv,
                             scoring='accuracy', n_jobs=-1)
    cv_results.append({'Model': name, 'CV Mean': scores.mean(),
                        'CV Std': scores.std(), 'CV Scores': scores})
    print(f'{name:<25} {scores.mean():.4f} ± {scores.std():.4f}')
    print(f'  Fold scores: {np.round(scores, 4)}')

# Boxplot
fig, ax = plt.subplots(figsize=(9, 5))
data_to_plot = [r['CV Scores'] for r in cv_results]
bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False)
colors = sns.color_palette('Set2', 3)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xticklabels(top3_names, rotation=15, ha='right')
ax.set_ylabel('CV Accuracy')
ax.set_title('5-Fold CV Distribution — Top 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hyperparameter Tuning with GridSearchCV

Tuning the best model (usually Random Forest or SVM).

In [ ]:
# Pick the model with the highest CV mean for tuning
best_cv_name = max(cv_results, key=lambda r: r['CV Mean'])['Model']
print(f'🔧 Tuning: {best_cv_name}')

if 'SVM' in best_cv_name:
    param_grid = {
        'C':      [0.1, 1, 10, 100],
        'gamma':  ['scale', 'auto', 0.001, 0.01],
        'kernel': ['rbf', 'poly']
    }
    base_clf = SVC(probability=True, random_state=42)
elif 'Random Forest' in best_cv_name:
    param_grid = {
        'n_estimators':      [100, 200, 300],
        'max_depth':         [None, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf':  [1, 2],
    }
    base_clf = RandomForestClassifier(random_state=42)
else:
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth': [3, 5]
    }
    base_clf = GradientBoostingClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator  = base_clf,
    param_grid = param_grid,
    cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring    = 'accuracy',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train_sc, y_train)

print(f'\n✅ Best Parameters : {grid_search.best_params_}')
print(f'   Best CV Score   : {grid_search.best_score_:.4f}')

In [ ]:
# ── Visualise GridSearch Results ──────────────────────────────────────────────
cv_df = pd.DataFrame(grid_search.cv_results_)
top_n = cv_df.nlargest(15, 'mean_test_score')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(top_n)), top_n['mean_test_score'],
       yerr=top_n['std_test_score'], capsize=4,
       color='mediumseagreen', edgecolor='black', alpha=0.8)
ax.axhline(grid_search.best_score_, color='red', linestyle='--',
           label=f'Best: {grid_search.best_score_:.4f}')
ax.set_xlabel('Parameter Combination Rank')
ax.set_ylabel('Mean CV Accuracy')
ax.set_title(f'GridSearchCV Results — {best_cv_name}', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('gridsearch_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Evaluation — Best Model

In [ ]:
best_model = grid_search.best_estimator_
y_pred     = best_model.predict(X_test_sc)
test_acc   = accuracy_score(y_test, y_pred)

class_labels = [c.replace('_', ' ').title() for c in CLASSES]

print('=' * 55)
print(f'  FINAL TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)')
print('=' * 55)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_labels))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Normalised %
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_labels)
disp2.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-class Accuracy Bar ────────────────────────────────────────────────────
per_class_acc = cm_norm.diagonal()
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(class_labels, per_class_acc,
              color=sns.color_palette('Set2', 4), edgecolor='black')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height()*100:.1f}%', ha='center', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.15)
ax.set_title('Per-Class Accuracy', fontsize=14, fontweight='bold')
ax.axhline(test_acc, color='red', linestyle='--', label=f'Overall: {test_acc:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model & Scaler with Pickle

In [ ]:
MODEL_PATH  = 'cricket_shot_model.pkl'
SCALER_PATH = 'cricket_shot_scaler.pkl'

with open(MODEL_PATH,  'wb') as f:
    pickle.dump(best_model, f)

with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)

print(f'✅ Model  saved → {MODEL_PATH}')
print(f'✅ Scaler saved → {SCALER_PATH}')

# ── Reload & Verify ───────────────────────────────────────────────────────────
with open(MODEL_PATH,  'rb') as f:
    loaded_model  = pickle.load(f)
with open(SCALER_PATH, 'rb') as f:
    loaded_scaler = pickle.load(f)

verify_acc = accuracy_score(y_test, loaded_model.predict(loaded_scaler.transform(X_test)))
print(f'\n🔁 Reloaded model test accuracy: {verify_acc:.4f}  (should match {test_acc:.4f})')
print('\n' + '='*55)
print('  PROJECT SUMMARY')
print('='*55)
print(f'  Dataset     : Cricket Shot Dataset (Kaggle)')
print(f'  Classes     : {CLASSES}')
print(f'  Best Model  : {best_cv_name}')
print(f'  Best Params : {grid_search.best_params_}')
print(f'  Test Acc    : {test_acc*100:.2f}%')
print(f'  Model saved : {MODEL_PATH}')
print(f'  Scaler saved: {SCALER_PATH}')
print('='*55)